In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, json
dir = os.getcwd()

ext = ['', '/..', '/../..', '/../../src/models', '/../../src/nlp', '/../../src/synth']
sys.path += [dir + i for i in ext]

In [2]:
from eval import *
from state import *
from expectation import *

In [3]:
from api.football import *

In [4]:
# Get folder path
folder_path = os.path.join('data')

demos = []
for i in os.listdir(folder_path):
    if i.startswith('demonstration4'): demos += [i]
    
demos.sort()

print(demos)

['demonstration4']


In [5]:
import json
from scene import Scene
from api.objects.registry import REGISTRY

scenes = {}

i = 0
for d in demos:
    if not i: # i=0 for first demo
        print('initial demo (w/ language)')
    else:
        print(f'helper demo {i}')

    folder = os.path.join(folder_path, d, 'json_segments')
    dir_list = [i for i in os.listdir(folder) if i.endswith('.json')]
    dir_list.sort()

    # print(dir_list)

    j = 0
    for f in dir_list:

        file = os.path.join(folder, f)

        # print(file)

        with open(file) as f:
            data = json.load(f)
        
        # for t in data['scene']:
        #     print(t)
        
        if not i:
            scenes[j] = Scene.from_dict(data['scene'], REGISTRY)
        else:
            try:
                scenes[j].add_demo(Scene.from_dict(data['scene'], REGISTRY))
            except KeyError:
                print(f"Attempted to access part {j} of a demonstration. Either non-existant or not in the original demo.")

        print(f'part {j} -> {scenes[j]}')

        j += 1
    

    i += 1
    print()

initial demo (w/ language)
part 0 -> <scene.Scene object at 0x12876f750>



In [6]:
objects = {obj.id: obj for obj in scenes[0].allObjects}

for id in objects.keys():
    print(id)

corner1
corner2
corner3
corner4
coach
leftBack
rightBack
Midfielder
CenterBack
opponent_A
opponent_B
opponent_C
opponent_D
opponent_E
goalkeeper
ball
goal
goal_leftpost
goal_rightpost
target


In [7]:
scenes[0].language

''

In [8]:
print(str(objects['corner1'].position))
print(str(objects['corner2'].position))
print(str(objects['corner3'].position))
print(str(objects['corner4'].position))
print(str(objects['goal']._position[0]))
print(str(objects['goal_leftpost']._position[0]))
print(str(objects['goal_rightpost']._position[0]))
print(str(objects['leftBack']._position[0]))

(x: -6.46, y: 11.04)
(x: -6.46, y: -11.04)
(x: 6.46, y: -11.021)
(x: 6.46, y: 11.04)
(x: 0.0, y: 16.0)
(x: 1.8745, y: 15.4685)
(x: -1.8745, y: 15.4685)
(x: -7.0, y: -8.500005)


In [9]:
print(str(objects['Midfielder']._position[0]))

(x: -3.0044086, y: 2.00061512)


In [10]:
left_back_has_ball = HasBallPosession(objects['leftBack'])
mid_into_channel = MovedToBox(objects['Midfielder'], (-10, -4), (-16, 16))
coach_moved_down = MovedToBox(objects['coach'], (-10, 10), (-16, -4))
coach_has_ball = HasBallPosession(objects['coach'])

In [11]:
from expectation import *

coach_received_ball = DidHappen([
    left_back_has_ball,
    (left_back_has_ball, False),
    coach_has_ball
])

coach_ever_down = DidHappen(coach_moved_down)

In [12]:
mid_has_ball = HasBallPosession(objects['Midfielder'])
mid_into_center = MovedToBox(objects['Midfielder'], (0, 0), (0, 0))

In [13]:
pass_forward_to_mid = Eventually(
    before=[
        coach_has_ball
    ],
    after=[
        (coach_has_ball, False),
        mid_has_ball
    ]
)

In [14]:
from eval import Eval
eval = Eval(scenes)

In [15]:
eval.sub([
    left_back_has_ball,
    mid_into_channel,
    coach_moved_down,
    coach_has_ball,
    mid_has_ball
])

eval.verify([
    coach_received_ball,
    coach_ever_down,
    pass_forward_to_mid
])

In [16]:
score = eval.run()

[0, 1, 0]
Score: 1/3


In [17]:
eval.timeline

[(coach moved to bounds within ((-10, 10), (-16, -4)), True),
 (leftBack has ball posession, True)]